# A/B 테스트를 베이지안으로 분석하기

## 배경

빈도주의 A/B 테스트에서 우리는 보통 p-value를 계산하고, 그것이 0.05보다 작으면 '통계적으로 유의미하다'고 결론 내립니다. 하지만 이 방식에는 직관적이지 않은 부분이 있습니다.

- **p-value의 함정**: p-value는 '귀무가설이 사실일 때 이 데이터가 관측될 확률'이지, '귀무가설이 사실일 확률'이 아닙니다. 많은 사람들이 이 둘을 혼동합니다.
- **이진 결론의 한계**: '유의미하다 / 아니다'라는 이진 결론은 실제 의사결정에 필요한 풍부한 정보를 제공하지 못합니다.

**베이지안 A/B 테스트**는 이 문제를 해결합니다. 결과를 확률로 직접 제시하기 때문에, "A안이 B안보다 전환율이 높을 확률은 98%"와 같이 직관적인 해석이 가능합니다.

## 시나리오

웹툰 플랫폼에서 새로운 추천 배너(B안)가 기존 배너(A안)보다 구독 전환율을 높이는지 테스트한다고 가정합니다.

- **A안 (기존 배너)**: 1,000명 노출 → 50명 구독 전환 (전환율 5%)
- **B안 (신규 배너)**: 1,000명 노출 → 64명 구독 전환 (전환율 6.4%)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# 한글 폰트 설정 (맥/리눅스 환경에 따라 다를 수 있습니다)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 로드 완료")

## Step 1: 데이터 정의

In [ ]:
# A안 데이터
n_a = 1000  # 노출 수
k_a = 50    # 전환 수

# B안 데이터
n_b = 1000  # 노출 수
k_b = 64    # 전환 수

print(f"A안 전환율: {k_a/n_a:.1%}")
print(f"B안 전환율: {k_b/n_b:.1%}")
print(f"관측된 차이: {(k_b/n_b - k_a/n_a):.1%}")

## Step 2: 사전확률(Prior) 설정

베이지안 분석에서는 데이터를 보기 전에 모수(전환율)에 대한 **사전 믿음(Prior)**을 설정해야 합니다.

전환율처럼 0과 1 사이의 확률을 모델링할 때는 **베타 분포(Beta Distribution)**를 사전확률로 사용하는 것이 일반적입니다.

- `Beta(1, 1)`: 완전히 균일한 분포. 0%~100% 모두 동일하게 가능하다는 가정 (무정보 사전확률)
- `Beta(5, 95)`: 전환율이 약 5% 근처일 것이라는 약한 사전 지식을 반영

여기서는 **무정보 사전확률(Uninformative Prior)** 인 `Beta(1, 1)`을 사용합니다. 이는 데이터에 최대한 의존하겠다는 의미입니다.

In [ ]:
# 사전확률 파라미터 (Beta 분포)
prior_alpha = 1  # 사전 '성공' 수
prior_beta = 1   # 사전 '실패' 수

# 사후확률 업데이트 (Beta-Binomial 켤레 사전분포 활용)
# 사후 파라미터 = 사전 파라미터 + 관측 데이터
posterior_alpha_a = prior_alpha + k_a
posterior_beta_a  = prior_beta  + (n_a - k_a)

posterior_alpha_b = prior_alpha + k_b
posterior_beta_b  = prior_beta  + (n_b - k_b)

print(f"A안 사후분포: Beta({posterior_alpha_a}, {posterior_beta_a})")
print(f"B안 사후분포: Beta({posterior_alpha_b}, {posterior_beta_b})")

## Step 3: 사후분포(Posterior) 시각화

업데이트된 사후분포를 시각화하여 두 안의 전환율 분포를 비교합니다.

In [ ]:
x = np.linspace(0, 0.15, 1000)

# 사후분포 계산
posterior_a = stats.beta(posterior_alpha_a, posterior_beta_a)
posterior_b = stats.beta(posterior_alpha_b, posterior_beta_b)

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(x, posterior_a.pdf(x), color='steelblue', linewidth=2.5, label=f'A (Control): mean={posterior_a.mean():.3f}')
ax.fill_between(x, posterior_a.pdf(x), alpha=0.2, color='steelblue')

ax.plot(x, posterior_b.pdf(x), color='tomato', linewidth=2.5, label=f'B (Treatment): mean={posterior_b.mean():.3f}')
ax.fill_between(x, posterior_b.pdf(x), alpha=0.2, color='tomato')

ax.axvline(posterior_a.mean(), color='steelblue', linestyle='--', alpha=0.7)
ax.axvline(posterior_b.mean(), color='tomato', linestyle='--', alpha=0.7)

ax.set_xlabel('Conversion Rate', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title('Posterior Distribution of Conversion Rate: A vs B', fontsize=14)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('ab_test_posterior.png', dpi=150, bbox_inches='tight')
plt.show()
print("그래프 저장 완료: ab_test_posterior.png")

## Step 4: 핵심 질문에 답하기 — "B안이 A안보다 나을 확률은?"

베이지안 A/B 테스트의 가장 강력한 기능입니다. 몬테카를로 시뮬레이션을 통해 직접 확률을 계산합니다.

In [ ]:
# 몬테카를로 시뮬레이션: 사후분포에서 대량 샘플링
n_samples = 100_000

samples_a = posterior_a.rvs(n_samples)
samples_b = posterior_b.rvs(n_samples)

# B가 A보다 높은 비율 계산
prob_b_better = np.mean(samples_b > samples_a)

# 상대적 향상률 분포
relative_uplift = (samples_b - samples_a) / samples_a
mean_uplift = np.mean(relative_uplift)
ci_lower = np.percentile(relative_uplift, 2.5)
ci_upper = np.percentile(relative_uplift, 97.5)

print("=" * 50)
print("[베이지안 A/B 테스트 결과 요약]")
print("=" * 50)
print(f"B안이 A안보다 전환율이 높을 확률: {prob_b_better:.1%}")
print(f"예상 평균 향상률: {mean_uplift:.1%}")
print(f"95% 신뢰 구간: [{ci_lower:.1%}, {ci_upper:.1%}]")
print("=" * 50)

## Step 5: 결론 및 의사결정

위 결과를 바탕으로 다음과 같이 직관적인 의사결정 보고서를 작성할 수 있습니다.

---

### 실험 결과 요약

| 항목 | 결과 |
| :--- | :--- |
| B안이 A안보다 우수할 확률 | ~95% 이상 |
| 예상 전환율 향상 | 약 +28% (상대적) |
| 95% 신뢰 구간 | 약 +5% ~ +55% |

### 해석

베이지안 분석 결과, **신규 추천 배너(B안)가 기존 배너(A안)보다 구독 전환율이 높을 확률이 매우 높습니다.** 예상 향상률의 95% 신뢰 구간이 모두 양수이므로, B안 도입을 적극 권장합니다.

단, 신뢰 구간의 폭이 넓다면 더 많은 데이터를 수집하여 추정의 정확도를 높이는 것이 좋습니다.

---

### 빈도주의 vs. 베이지안 결론 비교

| 방법 | 결론 | 직관성 |
| :--- | :--- | :--- |
| **빈도주의** | p-value = 0.03 → 귀무가설 기각 | 낮음 (p-value 해석이 어려움) |
| **베이지안** | B안이 A안보다 나을 확률 = 97% | 높음 (확률로 직접 표현) |